# The following codes focus on the

1) Merge multiple final dataframes into one combined file.

2) Add derived features (HR, BP, RR, SBP, DBP, etc.) and apply quality checks.

3) Remove low-quality PPG cases (based on SQI).

4) Exclude rows with mislabeled signals.

5) Stratify subjects by age, diagnosis codes, and other factors.

6) Build stratified folds for downstream training and evaluation.

## Merging the dataframe files to a create a merged file

In [ ]:
import os
import pandas as pd
import re

# Set your folder path
folder_path = 'path/to/folders/of/pkl/files'

# List and filter only .pkl files
files = os.listdir(folder_path)
pkl_files = [f for f in files if f.endswith('.pkl')]

# Sort files based on the numbers in the filename
def extract_numbers(f):
    match = re.search(r'df_(\d+)_(\d+)_\d+\_temp.pkl', f)
    return (int(match.group(1)), int(match.group(2))) if match else (float('inf'), float('inf'))



pkl_files_sorted = sorted(pkl_files, key=extract_numbers)

# Print the order of files to be merged
print("Order of files being merged:")
for f in pkl_files_sorted:
    print(f)

# Load and merge DataFrames
df_list = [pd.read_pickle(os.path.join(folder_path, f)) for f in pkl_files_sorted]
merged_df = pd.concat(df_list, ignore_index=True)

# Optional: Save the merged DataFrame
merged_df.to_pickle('path/to merged pkl file.pkl')

print("\nMerged DataFrame shape:", merged_df.shape)

## Now addig extra coloumns e.g., hr, bp, rr, sbp, dbp,..

In [ ]:
import pandas as pd
import numpy as np

df_events=df1


# Helper function to check if all elements in a list are not nan
def all_not_nan(x):
    return isinstance(x, (list, np.ndarray)) and all(not pd.isna(i) for i in x)

# Create new columns
df_events['bp'] = df_events.apply(lambda row: int(all_not_nan(row['vector_10s_sbp']) and all_not_nan(row['vector_10s_dbp'])), axis=1)
df_events['hr'] = df_events.apply(lambda row: int(all_not_nan(row['vector_10s_hr'])), axis=1)
df_events['rr'] = df_events['median_30s_rr'].apply(lambda x: int(not pd.isna(x)))



df=df_events

def strict_median(x):
    if isinstance(x, (list, np.ndarray)) and not any(pd.isna(x)):
        return np.median(x)
    return np.nan

def strict_iqr(x):
    if isinstance(x, (list, np.ndarray)) and not any(pd.isna(x)):
        return np.percentile(x, 75) - np.percentile(x, 25)
    return np.nan

# Apply to SBP
df['median_30s_sbp'] = df['vector_10s_sbp'].apply(strict_median)
df['iqr_30s_sbp'] = df['vector_10s_sbp'].apply(strict_iqr)

# Apply to DBP
df['median_30s_dbp'] = df['vector_10s_dbp'].apply(strict_median)
df['iqr_30s_dbp'] = df['vector_10s_dbp'].apply(strict_iqr)

# Apply to HR
df['median_30s_hr'] = df['vector_10s_hr'].apply(strict_median)
df['iqr_30s_hr'] = df['vector_10s_hr'].apply(strict_iqr)

## Removing the ppg segments which has negetive sqi 


In [ ]:
import numpy as np
df_events2=df_merged
# 3. Check if all values in vector_10s_pleth_sqi are 0 or 1
def is_valid_sqi(triple):
    triple_array = np.array(triple)
    return np.all(np.isin(triple_array, [0, 1])) if isinstance(triple, (list, tuple)) else False
mask = (
    (df_events2['vector_10s_pleth_sqi'].apply(is_valid_sqi))
)

# 5. Apply mask
df_events = df_events2[mask].copy()
df_events

## Startifying


In [ ]:
import numpy as np
import pandas as pd
from stratify import *

np.random.seed(42)

# =========================================================
# 1️⃣ Define important rhythms
# =========================================================

## we wanted to be sure that we will have these important rhythms in all of folds

important_rhythms = [
    "SR", "STACH", "AF", "SBRAD", "VPACE",
    "AVPACE", "AFLT", "APACE",
    "SARRH", "JR", "SVTACH",
    "MATACH", "VTACH"
]

# =========================================================
# 2️⃣ Create rhythm_for_strat (DO NOT DELETE ANY ROWS)
# =========================================================

df_events["rhythm_for_strat"] = df_events["event_rhythm"].where(
    df_events["event_rhythm"].isin(important_rhythms),
    "OTHER_STRAT"
)

# =========================================================
# 3️⃣ Age binning
# =========================================================

bins = [0, 18, 35, 50, 65, 85, np.inf]
labels = ['0-18', '19-35', '36-50', '51-65', '66-85', '85+']

df_events["age_binned"] = pd.cut(
    df_events["age"],
    bins=bins,
    labels=labels,
    right=False
)

# =========================================================
# 4️⃣ Safe cardiovascular ICD filtering
# =========================================================

def filter_cardiovascular_codes(icd_list):
    if not isinstance(icd_list, list):
        return []
    return [code for code in icd_list if isinstance(code, str) and code.startswith("I")]

df_events["cv_icd10"] = df_events["icd10_truncated"].apply(filter_cardiovascular_codes)

# =========================================================
# 5️⃣ Build FINAL stratification label
#     IMPORTANT: use rhythm_for_strat (NOT event_rhythm)
# =========================================================

df_events["label_for_stratification"] = df_events.apply(
    lambda row: [
        row["rhythm_for_strat"],
        str(row["age_binned"]),
        row["gender"]
    ] + row["cv_icd10"],
    axis=1
)

# =========================================================
# 6️⃣ Perform 10-fold patient-level stratification
# =========================================================

strat_folds = stratified_subsets(
    df_events,
    "label_for_stratification",
    [0.1] * 10,              # 10 folds
    col_group="subject_id",  # patient-level split
    random_seed=42
)

df_events["strat_fold"] = strat_folds

# =========================================================
# 7️⃣ Verification: ensure all important rhythms
#     appear in every fold
# =========================================================

verification = (
    df_events[df_events["event_rhythm"].isin(important_rhythms)]
    .groupby(["strat_fold", "event_rhythm"])["subject_id"]
    .nunique()
    .unstack(fill_value=0)
)

print("\nVerification table (unique subjects per fold & rhythm):\n")
print(verification)

# Optional: check for any missing rhythm in any fold
if (verification == 0).any().any():
    print("\n⚠️ WARNING: Some fold is missing at least one important rhythm.")
else:
    print("\n✅ All important rhythms are present in every fold.")


In [ ]:
df_events

Arbitrary filtering for BP and RR values

In [ ]:
#For segments where BP annotations were available, we verified that the median systolic and diastolic 
# pressures fell within physiological bounds, recommended by \citex{sanches2024mimic,ritchie2013extreme}:
#  SBP and DBP between 60–200~mmHg and  30–120~mmHg, respectively

to_delete_mask = (
    (df_events["bp"] == 1) &
    ~(
        df_events["median_30s_sbp"].between(60, 200, inclusive="both") &
        df_events["median_30s_dbp"].between(30, 120, inclusive="both")
    )
)
num_delete = to_delete_mask.sum()
total_bp_cases = (df_events["bp"] == 1).sum()
print("Number of BP==1 rows to be deleted:", num_delete)

After applying the physiological plausibility filters, 20,328 rows with valid BP labels were excluded. Given that the dataset contains more than 6.3 million rows, this represents only a negligible proportion of the data.

In [ ]:
# For 30-second segments with a RESP channel, the fundamental frequency was calculated from the RESP 
# signal. Segments in which the RESP signal exhibited a dominant frequency above 1 Hz were excluded, 
# since such values are not within physiologically plausible respiratory rates

import numpy as np

df_events["f_peak"] = np.where(
    df_events["median_30s_rr"].notna(),
    df_events["median_30s_rr"] / 60.0,
    np.nan
)
num_high = (
    df_events["f_peak"].notna() &
    (df_events["f_peak"] > 1.0)
).sum()
print("Number of rows with f_peak > 1 Hz:", num_high)
total_valid = df_events["f_peak"].notna().sum()

In total, 1,352 rows with respiratory peak frequencies exceeding 1 Hz were excluded. Considering the dataset contains more than 6.3 million rows, this corresponds to a negligible proportion of the data.